## 구간 분할
  - 연속형 데이터를 범주형 데이터로 사용할때  필요한 작업이다

pd.cut()은 값의 범위를 기준으로 나누고,    
pd.qcut()은 **데이터 분포(분위수)**를 기준으로 나눈다.  

| 항목           | `pd.cut()`              | `pd.qcut()`                  |
| ------------ | ----------------------- | ---------------------------- |
| 나누는 기준    | **값의 범위 (equal width)** | **데이터 개수 (equal frequency)** |
| 각 구간의 개수  | **다를 수 있음**             | **거의 동일함** (데이터 수 기준)        |
| 데이터 분포 고려 | 고려하지 않음               | 분위수 기반으로 분포 고려함            |
|  사용 예시     | 연령대를 10살 단위로 나누기 등      | 상위 25%, 중간 50%, 하위 25% 등     |

labels=False를 쓰면 정수형으로 범주값이 출력된다.  
right=False를 쓰면 구간이 왼쪽 포함 [ ) 방식이 된다.  
retbins=True를 쓰면 구간분할 영역을 리턴해 준다.

1. 고정 구간 (Equal-Width Binning)
  - 데이터 범위를 일정한 간격으로 나누는 방식

In [ ]:
import pandas as pd
import numpy as np

# 예시 데이터
data = pd.DataFrame({
    'age': [18, 25, 32, 40, 52, 60, 70, 85]
})

# 고정 구간으로 나누기 (예: 4개 구간)
data['age_bin'], bin_dividers= pd.cut(data['age'], bins=4, labels=False, retbins=True) #right: True
print(bin_dividers)
print(data)

[17.933 34.75  51.5   68.25  85.   ]
   age  age_bin
0   18        0
1   25        0
2   32        0
3   40        1
4   52        2
5   60        2
6   70        3
7   85        3


2. 동등 분포 구간 (Equal-Frequency / Quantile Binning)
- 각 구간에 데이터가 거의 같은 수로 들어가도록 분할

In [ ]:
# 데이터 수가 동일하게 분포되도록 4개 구간으로 나눔
data['age_qbin'] , bin_dividers= pd.qcut(data['age'], q=4, labels=False, retbins=True)
print(bin_dividers)
print(data)


[18.   30.25 46.   62.5  85.  ]
   age  age_bin  age_qbin
0   18        0         0
1   25        0         0
2   32        0         1
3   40        1         1
4   52        2         2
5   60        2         2
6   70        3         3
7   85        3         3


3. 사용자 정의 구간 지정 (Custom Binning)





In [ ]:
# 구간 수동 정의
bins = [0, 30, 50, 70, 100]
labels = ['청년', '중년', '장년', '노년']

data['age_group'] , bin_dividers= pd.cut(data['age'], bins=bins, labels=labels, right=False, retbins=True)
print(bin_dividers)
print(data)


[  0  30  50  70 100]
   age  age_bin  age_qbin age_group
0   18        0         0        청년
1   25        0         0        청년
2   32        0         1        중년
3   40        1         1        중년
4   52        2         2        장년
5   60        2         2        장년
6   70        3         3        노년
7   85        3         3        노년


## 더미변수

In [ ]:
import pandas as pd

# 예시 연속형 데이터
df = pd.DataFrame({'age': [18, 25, 32, 40, 52, 60, 70, 85]})

# 1단계: 구간으로 나누기
df['age_bin'] , bin_dividers= pd.cut(df['age'], bins=[0, 30, 50, 70, 100], labels=['청년', '중년', '장년', '노년'], retbins=True)
print(bin_dividers)
print(df)

# 2단계: 더미 변수로 변환
age_dummies = pd.get_dummies(df['age_bin'], prefix='age')

# 결과 병합
df = pd.concat([df, age_dummies], axis=1)

print(df)

[  0  30  50  70 100]
   age age_bin
0   18      청년
1   25      청년
2   32      중년
3   40      중년
4   52      장년
5   60      장년
6   70      장년
7   85      노년
   age age_bin  age_청년  age_중년  age_장년  age_노년
0   18      청년    True   False   False   False
1   25      청년    True   False   False   False
2   32      중년   False    True   False   False
3   40      중년   False    True   False   False
4   52      장년   False   False    True   False
5   60      장년   False   False    True   False
6   70      장년   False   False    True   False
7   85      노년   False   False   False    True


| `strategy` 값 | 설명                    | 예시 구간 분포                          |
| ------------ | --------------------- | --------------------------------- |
| `'uniform'`  | **값 범위 기준** 균등 분할     | `pd.cut()`과 유사 (equal-width)      |
| `'quantile'` | **데이터 수 기준** 균등 분할    | `pd.qcut()`과 유사 (equal-frequency) |
| `'kmeans'`   | K-means 클러스터링으로 구간 설정 | 데이터 분포를 고려한 최적 분할                 |


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
import pandas as pd
import numpy as np

# 예시 데이터
X = np.array([[18], [25], [32], [40], [52], [60], [70], [85]])

# KBinsDiscretizer 사용 (원-핫 인코딩)
# strategy='uniform' : 전체 최솟값~최댓값 범위를 n_bins개의 동일한 길이로 분할
kbd = KBinsDiscretizer(n_bins=4, encode='onehot-dense', strategy='uniform')
X_binned = kbd.fit_transform(X)

# 결과 출력
df = pd.DataFrame(X_binned, columns=[f'bin_{i}' for i in range(X_binned.shape[1])])
df['age'] = X.flatten()

print(df)


   bin_0  bin_1  bin_2  bin_3  age
0    1.0    0.0    0.0    0.0   18
1    1.0    0.0    0.0    0.0   25
2    1.0    0.0    0.0    0.0   32
3    0.0    1.0    0.0    0.0   40
4    0.0    0.0    1.0    0.0   52
5    0.0    0.0    1.0    0.0   60
6    0.0    0.0    0.0    1.0   70
7    0.0    0.0    0.0    1.0   85


In [ ]:
# LabelEncoder  : 카테고리형 데이터 → 숫자로 변환
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(["red", "blue", "green"])
y

array([2, 0, 1])

In [ ]:
# OneHotEncoder : 카테고리형 데이터 → 원-핫 벡터로 변환
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
X = encoder.fit_transform([["red"], ["blue"], ["green"]])
X


array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]])

In [ ]:
# OrdinalEncoder : 카테고리형 데이터를 순서형 숫자로 변환
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
X = encoder.fit_transform([["low"], ["medium"], ["high"]])
X


array([[1.],
       [2.],
       [0.]])